 SP-XGBoost: Feature Engineering & Dataset Preparation
 
**Objective**: Encode categorical variables in already-split datasets from Notebook 1


In [ ]:
# ===========================================
# Import Libraries
# Notebook 2: Feature Engineering
# ===========================================
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns

# Reproducibility
SEED = 42
np.random.seed(SEED)

# Display options
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)
print("✅ Libraries imported successfully")
# Project directories
PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "data"
FIELD_DIR = DATA_DIR / "field"
PROCESSED_DIR = DATA_DIR / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURE_DIR = RESULTS_DIR / "figures"
TABLE_DIR = RESULTS_DIR / "tables"
REPORT_DIR = RESULTS_DIR / "reports"
# Create output directories
for folder in [FIGURE_DIR, TABLE_DIR, REPORT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

✅ Libraries imported successfully


In [41]:
#Load all four datasets
# ============================================================
# LOAD NOTEBOOK 1 OUTPUTS
# ============================================================

train_binary = pd.read_csv(PROCESSED_DIR / "train_binary.csv")
test_binary = pd.read_csv(PROCESSED_DIR / "test_binary.csv")

train_multiclass = pd.read_csv(PROCESSED_DIR / "train_multiclass.csv")
test_multiclass = pd.read_csv(PROCESSED_DIR / "test_multiclass.csv")

print("=" * 70)
print("DATASETS LOADED")
print("=" * 70)

print(f"Binary training:     {train_binary.shape}")
print(f"Binary testing:      {test_binary.shape}")
print(f"Multiclass training: {train_multiclass.shape}")
print(f"Multiclass testing:  {test_multiclass.shape}")

DATASETS LOADED
Binary training:     (468, 62)
Binary testing:      (117, 62)
Multiclass training: (468, 62)
Multiclass testing:  (117, 62)


In [42]:
#CELL 3-Verify the Notebook 1 outputs
# ============================================================
# VERIFY EXPECTED DATASET DIMENSIONS
# ============================================================

expected_shapes = {
    "train_binary": (468, 62),
    "test_binary": (117, 62),
    "train_multiclass": (468, 62),
    "test_multiclass": (117, 62)
}

actual_shapes = {
    "train_binary": train_binary.shape,
    "test_binary": test_binary.shape,
    "train_multiclass": train_multiclass.shape,
    "test_multiclass": test_multiclass.shape
}

print("=" * 70)
print("DATASET VALIDATION")
print("=" * 70)

all_correct = True

for name, expected in expected_shapes.items():
    actual = actual_shapes[name]

    if actual == expected:
        print(f"✓ {name:<20} {actual}")
    else:
        print(f"✗ {name:<20} Expected {expected}, got {actual}")
        all_correct = False

if all_correct:
    print("\n✓ ALL DATASET DIMENSIONS ARE CORRECT")
else:
    print("\n✗ DATASET VALIDATION FAILED — STOP")
    raise ValueError("One or more datasets have unexpected dimensions.")

DATASET VALIDATION
✓ train_binary         (468, 62)
✓ test_binary          (117, 62)
✓ train_multiclass     (468, 62)
✓ test_multiclass      (117, 62)

✓ ALL DATASET DIMENSIONS ARE CORRECT


In [43]:
# ============================================================
#CELL 4- VERIFY TARGET VARIABLES
# ============================================================

required_targets = ["RISK_LABEL", "RISK_BINARY"]

print("=" * 70)
print("TARGET VARIABLE VALIDATION")
print("=" * 70)

for dataset_name, df in {
    "train_binary": train_binary,
    "test_binary": test_binary,
    "train_multiclass": train_multiclass,
    "test_multiclass": test_multiclass
}.items():

    missing_targets = [
        target for target in required_targets
        if target not in df.columns
    ]

    if missing_targets:
        print(f"✗ {dataset_name}: Missing {missing_targets}")
        raise ValueError(
            f"{dataset_name} does not contain all required target variables."
        )

    print(f"✓ {dataset_name}: RISK_LABEL and RISK_BINARY present")

print("\n✓ TARGET VALIDATION PASSED")

TARGET VARIABLE VALIDATION
✓ train_binary: RISK_LABEL and RISK_BINARY present
✓ test_binary: RISK_LABEL and RISK_BINARY present
✓ train_multiclass: RISK_LABEL and RISK_BINARY present
✓ test_multiclass: RISK_LABEL and RISK_BINARY present

✓ TARGET VALIDATION PASSED


In [44]:
#Cell 5 — Define target and identifier columns
# ============================================================
# DEFINE TARGETS AND NON-PREDICTOR COLUMNS
# ============================================================

ID_COL = "SID"

PRIMARY_TARGET = "RISK_BINARY"
SECONDARY_TARGET = "RISK_LABEL"

TARGET_COLS = [
    PRIMARY_TARGET,
    SECONDARY_TARGET
]

NON_PREDICTOR_COLS = [
    ID_COL,
    PRIMARY_TARGET,
    SECONDARY_TARGET
]

print("=" * 70)
print("PREDICTOR / OUTCOME CONFIGURATION")
print("=" * 70)

print(f"Identifier:          {ID_COL}")
print(f"Primary outcome:     {PRIMARY_TARGET}")
print(f"Secondary outcome:   {SECONDARY_TARGET}")
print(f"Non-predictors:      {NON_PREDICTOR_COLS}")

PREDICTOR / OUTCOME CONFIGURATION
Identifier:          SID
Primary outcome:     RISK_BINARY
Secondary outcome:   RISK_LABEL
Non-predictors:      ['SID', 'RISK_BINARY', 'RISK_LABEL']


In [45]:
# ============================================================
# CELL 6- CONSTRUCT PREDICTOR SET
# ============================================================

predictor_cols = [
    col for col in train_binary.columns
    if col not in NON_PREDICTOR_COLS
]

print("=" * 70)
print("PREDICTOR SET")
print("=" * 70)

print(f"Number of predictors: {len(predictor_cols)}")

for i, col in enumerate(predictor_cols, start=1):
    print(f"{i:02d}. {col}")

PREDICTOR SET
Number of predictors: 59
01. GPA_S1
02. CA_AVG
03. EXAM_AVG
04. CLIN_AVG
05. LAB_AVG
06. ATT_RATE
07. ASSIGN_LATE
08. Age_group
09. Gender
10. Year_study
11. Programme
12. SES
13. Financial_diff
14. Employment_hrs
15. Study_hrs_day
16. Sleep_hrs
17. Self_risk_percep
18. Reviews notes within 24h
19. Understands content pre-exam
20. Seeks help when stuck
21. Uses library regularly
22. Completes readings
23. Takes organised notes
24. Concentration in self-study
25. Participates in class
26. Clinical takes study time
27. Prepared for clinical assess
28. Rotations affect performance
29. Adequate supervision
30. Schedule conflicts
31. Confident in clinical skills
32. Anxious about assessments
33. Sleep difficulty
34. Burnt out
35. Hopeless/unmotivated
36. Physical health affected
37. Considered break
38. Emotionally supported
39. Lecturers approachable
40. Sleep affects concentration
41. Regular exercise
42. Balanced diet
43. Health interferes studies
44. Takes rest breaks
45. 

In [46]:
# ============================================================
# CELL: 7 - VERIFY PREDICTOR COUNT
# ============================================================

expected_predictors = 59

print("=" * 70)
print("PREDICTOR COUNT VALIDATION")
print("=" * 70)

print(f"Total columns:       {train_binary.shape[1]}")
print(f"Non-predictors:      {len(NON_PREDICTOR_COLS)}")
print(f"Candidate predictors: {len(predictor_cols)}")
print(f"Expected predictors:  {expected_predictors}")

if len(predictor_cols) == expected_predictors:
    print("\n✓ PREDICTOR COUNT VALIDATED")
else:
    raise ValueError(
        f"Expected {expected_predictors} predictors, "
        f"but found {len(predictor_cols)}."
    )

PREDICTOR COUNT VALIDATION
Total columns:       62
Non-predictors:      3
Candidate predictors: 59
Expected predictors:  59

✓ PREDICTOR COUNT VALIDATED


In [47]:
# ============================================================
# CELL 8- DISPLAY ALL CANDIDATE PREDICTORS
# ============================================================

print("=" * 70)
print("ALL 59 CANDIDATE PREDICTORS")
print("=" * 70)

for i, col in enumerate(predictor_cols, start=1):
    print(f"{i:02d}. {col}")

print("=" * 70)
print(f"Total candidate predictors: {len(predictor_cols)}")


ALL 59 CANDIDATE PREDICTORS
01. GPA_S1
02. CA_AVG
03. EXAM_AVG
04. CLIN_AVG
05. LAB_AVG
06. ATT_RATE
07. ASSIGN_LATE
08. Age_group
09. Gender
10. Year_study
11. Programme
12. SES
13. Financial_diff
14. Employment_hrs
15. Study_hrs_day
16. Sleep_hrs
17. Self_risk_percep
18. Reviews notes within 24h
19. Understands content pre-exam
20. Seeks help when stuck
21. Uses library regularly
22. Completes readings
23. Takes organised notes
24. Concentration in self-study
25. Participates in class
26. Clinical takes study time
27. Prepared for clinical assess
28. Rotations affect performance
29. Adequate supervision
30. Schedule conflicts
31. Confident in clinical skills
32. Anxious about assessments
33. Sleep difficulty
34. Burnt out
35. Hopeless/unmotivated
36. Physical health affected
37. Considered break
38. Emotionally supported
39. Lecturers approachable
40. Sleep affects concentration
41. Regular exercise
42. Balanced diet
43. Health interferes studies
44. Takes rest breaks
45. Sense of be

In [48]:
# ============================================================
# CELL 9 --DISPLAY PREDICTORS IN SMALL GROUPS
# ============================================================

print("=" * 70)
print("CANDIDATE PREDICTORS — GROUPED OUTPUT")
print("=" * 70)

for start in range(0, len(predictor_cols), 10):
    end = min(start + 10, len(predictor_cols))

    print(f"\n--- Predictors {start + 1}–{end} ---")

    for i in range(start, end):
        print(f"{i + 1:02d}. {predictor_cols[i]}")

print("\n" + "=" * 70)
print(f"TOTAL: {len(predictor_cols)} CANDIDATE PREDICTORS")
print("=" * 70)

CANDIDATE PREDICTORS — GROUPED OUTPUT

--- Predictors 1–10 ---
01. GPA_S1
02. CA_AVG
03. EXAM_AVG
04. CLIN_AVG
05. LAB_AVG
06. ATT_RATE
07. ASSIGN_LATE
08. Age_group
09. Gender
10. Year_study

--- Predictors 11–20 ---
11. Programme
12. SES
13. Financial_diff
14. Employment_hrs
15. Study_hrs_day
16. Sleep_hrs
17. Self_risk_percep
18. Reviews notes within 24h
19. Understands content pre-exam
20. Seeks help when stuck

--- Predictors 21–30 ---
21. Uses library regularly
22. Completes readings
23. Takes organised notes
24. Concentration in self-study
25. Participates in class
26. Clinical takes study time
27. Prepared for clinical assess
28. Rotations affect performance
29. Adequate supervision
30. Schedule conflicts

--- Predictors 31–40 ---
31. Confident in clinical skills
32. Anxious about assessments
33. Sleep difficulty
34. Burnt out
35. Hopeless/unmotivated
36. Physical health affected
37. Considered break
38. Emotionally supported
39. Lecturers approachable
40. Sleep affects concent

In [49]:
# ============================================================
# CELL 10- TEMPORAL ELIGIBILITY AUDIT
# ============================================================

eligible_predictors = predictor_cols.copy()

print("=" * 70)
print("TEMPORAL ELIGIBILITY AUDIT")
print("=" * 70)

print(f"Candidate predictors: {len(predictor_cols)}")
print(f"Provisionally eligible predictors: {len(eligible_predictors)}")

print("\n✓ Early intervention provided retained")
print("  Reason: Semester-level information")
print("  Condition: Must represent information available by the")
print("  defined Semester 1 prediction point.")

# ============================================================
# CELL--VERIFY ALL CANDIDATE PREDICTORS ARE ACCOUNTED FOR
# ============================================================

missing_from_eligibility = [
    col for col in predictor_cols
    if col not in eligible_predictors
]

unexpected_variables = [
    col for col in eligible_predictors
    if col not in predictor_cols
]

print("=" * 70)
print("TEMPORAL AUDIT VALIDATION")
print("=" * 70)

print(f"Candidate predictors: {len(predictor_cols)}")
print(f"Eligible predictors:  {len(eligible_predictors)}")
print(f"Missing:              {len(missing_from_eligibility)}")
print(f"Unexpected:           {len(unexpected_variables)}")

if not missing_from_eligibility and not unexpected_variables:
    print("\n✓ ALL 59 CANDIDATE PREDICTORS ACCOUNTED FOR")
else:
    print("\n✗ TEMPORAL AUDIT VALIDATION FAILED")
    
    if missing_from_eligibility:
        print("\nMissing:")
        for col in missing_from_eligibility:
            print(f"  - {col}")
    
    if unexpected_variables:
        print("\nUnexpected:")
        for col in unexpected_variables:
            print(f"  - {col}")

TEMPORAL ELIGIBILITY AUDIT
Candidate predictors: 59
Provisionally eligible predictors: 59

✓ Early intervention provided retained
  Reason: Semester-level information
  Condition: Must represent information available by the
  defined Semester 1 prediction point.
TEMPORAL AUDIT VALIDATION
Candidate predictors: 59
Eligible predictors:  59
Missing:              0
Unexpected:           0

✓ ALL 59 CANDIDATE PREDICTORS ACCOUNTED FOR


In [50]:
# ============================================================
# CELL 11- MISSING-VALUE AUDIT — TRAINING DATA
# ============================================================

X_train_binary = train_binary[eligible_predictors].copy()

missing_train = X_train_binary.isnull().sum()
missing_train_pct = (missing_train / len(X_train_binary)) * 100

missing_summary_train = pd.DataFrame({
    "Missing_Count": missing_train,
    "Missing_Percent": missing_train_pct
})

missing_summary_train = (
    missing_summary_train[
        missing_summary_train["Missing_Count"] > 0
    ]
    .sort_values("Missing_Count", ascending=False)
)

print("=" * 70)
print("MISSING-VALUE AUDIT — TRAINING DATA")
print("=" * 70)

if missing_summary_train.empty:
    print("✓ No missing values detected in training predictors.")
else:
    print(missing_summary_train.to_string())

MISSING-VALUE AUDIT — TRAINING DATA
✓ No missing values detected in training predictors.


In [51]:
# ============================================================
# CELL 12-- MISSING-VALUE AUDIT — TESTING DATA
# ============================================================

X_test_binary = test_binary[eligible_predictors].copy()

missing_test = X_test_binary.isnull().sum()
missing_test_pct = (missing_test / len(X_test_binary)) * 100

missing_summary_test = pd.DataFrame({
    "Missing_Count": missing_test,
    "Missing_Percent": missing_test_pct
})

missing_summary_test = (
    missing_summary_test[
        missing_summary_test["Missing_Count"] > 0
    ]
    .sort_values("Missing_Count", ascending=False)
)

print("=" * 70)
print("MISSING-VALUE AUDIT — TESTING DATA")
print("=" * 70)

if missing_summary_test.empty:
    print("✓ No missing values detected in testing predictors.")
else:
    print(missing_summary_test.to_string())

MISSING-VALUE AUDIT — TESTING DATA
✓ No missing values detected in testing predictors.


In [ ]:
# ============================================================
# CELL 13--OVERALL MISSINGNESS SUMMARY
# ============================================================

total_train_missing = X_train_binary.isnull().sum().sum()
total_test_missing = X_test_binary.isnull().sum().sum()

print("=" * 70)
print("OVERALL MISSINGNESS")
print("=" * 70)

print(f"Training missing cells: {total_train_missing}")
print(f"Testing missing cells:  {total_test_missing}")

if total_train_missing == 0 and total_test_missing == 0:
    print("\n✓ COMPLETE-CASE PREDICTOR DATA")
else:
    print("\n⚠ Missing values detected.")
    print("Missing-value treatment will be determined after reviewing")
    print("the variable types and missingness pattern.")

OVERALL MISSINGNESS
Training missing cells: 0
Testing missing cells:  0

✓ COMPLETE-CASE PREDICTOR DATA


In [53]:
# ============================================================
# CELL 14 -DATA-TYPE AUDIT
# ============================================================

dtype_summary = pd.DataFrame({
    "Variable": eligible_predictors,
    "Data_Type": [
        train_binary[col].dtype
        for col in eligible_predictors
    ],
    "Unique_Values": [
        train_binary[col].nunique()
        for col in eligible_predictors
    ]
})

print("=" * 70)
print("DATA-TYPE AUDIT — 59 PREDICTORS")
print("=" * 70)

print(dtype_summary.to_string(index=False))

DATA-TYPE AUDIT — 59 PREDICTORS
                     Variable Data_Type  Unique_Values
                       GPA_S1   float64            171
                       CA_AVG   float64             43
                     EXAM_AVG   float64             44
                     CLIN_AVG   float64             43
                      LAB_AVG   float64             48
                     ATT_RATE   float64             32
                  ASSIGN_LATE    object              4
                    Age_group    object              5
                       Gender    object              2
                   Year_study     int64              2
                    Programme    object              2
                          SES    object              5
               Financial_diff    object              5
               Employment_hrs    object              4
                Study_hrs_day    object              5
                    Sleep_hrs    object              6
             Self_risk_percep    

In [54]:
# ============================================================
# CELL 15 -- UNIQUE-VALUE INSPECTION
# ============================================================

print("=" * 70)
print("UNIQUE VALUES — 59 PREDICTORS")
print("=" * 70)

for col in eligible_predictors:
    values = train_binary[col].dropna().unique()

    print(f"\n{col}")
    print(f"  dtype: {train_binary[col].dtype}")
    print(f"  n_unique: {len(values)}")
    print(f"  values: {values[:20]}")

UNIQUE VALUES — 59 PREDICTORS

GPA_S1
  dtype: float64
  n_unique: 171
  values: [2.78 2.28 2.76 2.52 3.07 3.35 2.9  2.8  2.26 3.19 2.95 2.86 0.98 3.08
 2.44 1.74 2.42 3.48 1.57 3.11]

CA_AVG
  dtype: float64
  n_unique: 43
  values: [0.88 0.96 0.79 0.69 0.89 0.7  0.91 0.57 1.   0.66 0.87 0.9  0.5  0.78
 0.99 0.93 0.92 0.77 0.67 0.75]

EXAM_AVG
  dtype: float64
  n_unique: 44
  values: [0.79 0.87 0.8  0.9  0.6  0.69 0.72 1.   0.63 0.7  0.92 0.66 0.89 0.77
 0.81 0.99 0.59 0.98 0.78 0.65]

CLIN_AVG
  dtype: float64
  n_unique: 43
  values: [0.89 0.66 0.6  0.7  0.79 0.85 0.9  0.69 0.77 0.92 0.91 0.68 1.   0.87
 0.81 0.93 0.57 0.99 0.8  0.78]

LAB_AVG
  dtype: float64
  n_unique: 48
  values: [0.97 1.   0.73 0.95 0.8  0.92 0.99 0.94 0.6  0.85 0.9  0.7  0.74 0.75
 0.93 0.47 0.89 0.78 0.96 0.91]

ATT_RATE
  dtype: float64
  n_unique: 32
  values: [0.79 0.7  0.89 0.9  0.93 0.99 0.72 0.94 0.96 0.57 1.   0.67 0.88 0.66
 0.69 0.78 0.6  0.73 0.92 0.85]

ASSIGN_LATE
  dtype: object
  n_unique: 4
 

In [55]:
# ============================================================
# CELL 16 --VARIABLE-TYPE CLASSIFICATION
# ============================================================

# ------------------------------------------------------------
# CONTINUOUS VARIABLES
# ------------------------------------------------------------

continuous_cols = [
    "GPA_S1",
    "CA_AVG",
    "EXAM_AVG",
    "CLIN_AVG",
    "LAB_AVG",
    "ATT_RATE"
]


# ------------------------------------------------------------
# NOMINAL CATEGORICAL VARIABLES
# ------------------------------------------------------------

nominal_cols = [
    "Gender",
    "Programme"
]


# ------------------------------------------------------------
# ORDINAL CATEGORICAL VARIABLES
# ------------------------------------------------------------

ordinal_cols = [
    "ASSIGN_LATE",
    "Age_group",
    "SES",
    "Financial_diff",
    "Employment_hrs",
    "Study_hrs_day",
    "Sleep_hrs",
    "Self_risk_percep"
]


# ------------------------------------------------------------
# LIKERT-SCALE VARIABLES
# ------------------------------------------------------------

likert_cols = [
    "Reviews notes within 24h",
    "Understands content pre-exam",
    "Seeks help when stuck",
    "Uses library regularly",
    "Completes readings",
    "Takes organised notes",
    "Concentration in self-study",
    "Participates in class",
    "Clinical takes study time",
    "Prepared for clinical assess",
    "Rotations affect performance",
    "Adequate supervision",
    "Schedule conflicts",
    "Confident in clinical skills",
    "Anxious about assessments",
    "Sleep difficulty",
    "Burnt out",
    "Hopeless/unmotivated",
    "Physical health affected",
    "Considered break",
    "Emotionally supported",
    "Lecturers approachable",
    "Sleep affects concentration",
    "Regular exercise",
    "Balanced diet",
    "Health interferes studies",
    "Takes rest breaks",
    "Sense of belonging",
    "Participates in clubs",
    "Adequate advisory support",
    "Concerns taken seriously",
    "Early intervention provided",
    "Peer study groups effective",
    "Knows where to get help",
    "Programme prepares for career",
    "Confident to complete prog",
    "Can improve performance",
    "Sets academic goals",
    "Persists when difficult",
    "Manages time effectively",
    "Self-motivates",
    "Feels prepared"
]


# ------------------------------------------------------------
# YEAR OF STUDY
# ------------------------------------------------------------

year_study_col = "Year_study"


# ============================================================
# VALIDATION
# ============================================================

classified_cols = (
    continuous_cols
    + nominal_cols
    + ordinal_cols
    + likert_cols
    + [year_study_col]
)

print("=" * 70)
print("VARIABLE-TYPE CLASSIFICATION")
print("=" * 70)

print(f"Continuous:     {len(continuous_cols)}")
print(f"Nominal:        {len(nominal_cols)}")
print(f"Ordinal:        {len(ordinal_cols)}")
print(f"Likert:         {len(likert_cols)}")
print(f"Year of study:  {1}")
print(f"TOTAL:          {len(classified_cols)}")

if set(classified_cols) == set(eligible_predictors):
    print("\n✓ ALL 59 PREDICTORS CLASSIFIED EXACTLY ONCE")
else:
    missing = set(eligible_predictors) - set(classified_cols)
    duplicated = [
        col for col in classified_cols
        if classified_cols.count(col) > 1
    ]

    print("\n✗ CLASSIFICATION VALIDATION FAILED")

    if missing:
        print("\nUnclassified:")
        for col in sorted(missing):
            print(f"  - {col}")

    if duplicated:
        print("\nDuplicated:")
        for col in sorted(set(duplicated)):
            print(f"  - {col}")

    raise ValueError("Predictor classification is incomplete or duplicated.")

VARIABLE-TYPE CLASSIFICATION
Continuous:     6
Nominal:        2
Ordinal:        8
Likert:         42
Year of study:  1
TOTAL:          59

✓ ALL 59 PREDICTORS CLASSIFIED EXACTLY ONCE


In [56]:
# ============================================================
# CELL 17 --EXPLICIT ORDINAL ENCODING MAPPINGS
# ============================================================

ordinal_mappings = {

    # Assignment submission lateness
    "ASSIGN_LATE": {
        "Almost never": 1,
        "Rarely": 2,
        "Sometimes": 3,
        "Always": 4
    },

    # Age group
    "Age_group": {
        "18-20 years": 1,
        "21-23 years": 2,
        "24-26 years": 3,
        "27-29 years": 4,
        "30-35 years": 5
    },

    # Socioeconomic status
    "SES": {
        "Very low income": 1,
        "Low income": 2,
        "Middle income": 3,
        "Upper-middle income": 4,
        "High income": 5
    },

    # Financial difficulty
    "Financial_diff": {
        "Never": 1,
        "Rarely (once or twice a semester)": 2,
        "Sometimes (monthly)": 3,
        "Often (weekly)": 4,
        "Always": 5
    },

    # Employment hours
    "Employment_hrs": {
        "No": 1,
        "Yes - fewer than 10 hours/week": 2,
        "Yes - 10-20 hours/week": 3,
        "Yes - more than 20 hours/week": 4
    },

    # Study hours per day
    "Study_hrs_day": {
        "Less than 1 hour": 1,
        "1-2 hours": 2,
        "2-3 hours": 3,
        "3-4 hours": 4,
        "More than 4 hours": 5
    },

    # Sleep hours
    "Sleep_hrs": {
        "Less than 4 hours": 1,
        "4-5 hours": 2,
        "5-6 hours": 3,
        "6-7 hours": 4,
        "7-8 hours": 5,
        "More than 8 hours": 6
    },

    # Perceived academic risk
    "Self_risk_percep": {
        "Very low risk": 1,
        "Low risk": 2,
        "Moderate risk": 3,
        "High risk": 4
    }
}

In [57]:
# CELL 18 & 19 Validate the mapping
# ============================================================
# CELL-18 YEAR OF STUDY MAPPING
# ============================================================

year_study_mapping = {
    200: 2,
    300: 3
}

print("=" * 70)
print("YEAR OF STUDY MAPPING")
print("=" * 70)

print("200 → Year 2")
print("300 → Year 3")

# ============================================================
# CELL-19 VALIDATE ORDINAL MAPPINGS
# ============================================================

print("=" * 70)
print("ORDINAL MAPPING VALIDATION")
print("=" * 70)

mapping_errors = []

for col, mapping in ordinal_mappings.items():

    actual_values = set(train_binary[col].dropna().unique())
    mapping_values = set(mapping.keys())

    missing_in_mapping = actual_values - mapping_values
    unused_mapping_values = mapping_values - actual_values

    print(f"\n{col}")
    print(f"  Dataset categories: {sorted(actual_values)}")
    print(f"  Mapping categories: {sorted(mapping_values)}")

    if missing_in_mapping:
        print(f"  ✗ Missing from mapping: {missing_in_mapping}")
        mapping_errors.append((col, "missing", missing_in_mapping))

    else:
        print("  ✓ All observed categories mapped")


# Validate Year_study separately

actual_years = set(train_binary["Year_study"].dropna().unique())
mapping_years = set(year_study_mapping.keys())

print("\nYear_study")
print(f"  Dataset values: {sorted(actual_years)}")
print(f"  Mapping values: {sorted(mapping_years)}")

if actual_years - mapping_years:
    print(f"  ✗ Missing from mapping: {actual_years - mapping_years}")
    mapping_errors.append(
        ("Year_study", "missing", actual_years - mapping_years)
    )
else:
    print("  ✓ All observed values mapped")


if mapping_errors:
    raise ValueError(
        "One or more ordinal mappings do not cover all observed categories."
    )

print("\n✓ ALL ORDINAL MAPPINGS VALIDATED")

YEAR OF STUDY MAPPING
200 → Year 2
300 → Year 3
ORDINAL MAPPING VALIDATION

ASSIGN_LATE
  Dataset categories: ['Almost never', 'Always', 'Rarely', 'Sometimes']
  Mapping categories: ['Almost never', 'Always', 'Rarely', 'Sometimes']
  ✓ All observed categories mapped

Age_group
  Dataset categories: ['18-20 years', '21-23 years', '24-26 years', '27-29 years', '30-35 years']
  Mapping categories: ['18-20 years', '21-23 years', '24-26 years', '27-29 years', '30-35 years']
  ✓ All observed categories mapped

SES
  Dataset categories: ['High income', 'Low income', 'Middle income', 'Upper-middle income', 'Very low income']
  Mapping categories: ['High income', 'Low income', 'Middle income', 'Upper-middle income', 'Very low income']
  ✓ All observed categories mapped

Financial_diff
  Dataset categories: ['Always', 'Never', 'Often (weekly)', 'Rarely (once or twice a semester)', 'Sometimes (monthly)']
  Mapping categories: ['Always', 'Never', 'Often (weekly)', 'Rarely (once or twice a semester

In [58]:
# ============================================================
# CELL 20 --CREATE PREPROCESSING COPIES
# ============================================================

X_train = train_binary[eligible_predictors].copy()
X_test = test_binary[eligible_predictors].copy()

y_train_binary = train_binary["RISK_BINARY"].copy()
y_test_binary = test_binary["RISK_BINARY"].copy()

y_train_multiclass = train_multiclass["RISK_LABEL"].copy()
y_test_multiclass = test_multiclass["RISK_LABEL"].copy()

print("=" * 70)
print("PREPROCESSING DATASETS CREATED")
print("=" * 70)

print(f"X_train:            {X_train.shape}")
print(f"X_test:             {X_test.shape}")
print(f"y_train_binary:     {y_train_binary.shape}")
print(f"y_test_binary:      {y_test_binary.shape}")
print(f"y_train_multiclass: {y_train_multiclass.shape}")
print(f"y_test_multiclass:  {y_test_multiclass.shape}")

PREPROCESSING DATASETS CREATED
X_train:            (468, 59)
X_test:             (117, 59)
y_train_binary:     (468,)
y_test_binary:      (117,)
y_train_multiclass: (468,)
y_test_multiclass:  (117,)


In [59]:
# ============================================================
# CELL 21 --APPLY ORDINAL MAPPINGS
# ============================================================

X_train_encoded = X_train.copy()
X_test_encoded = X_test.copy()

# Apply ordinal mappings
for col, mapping in ordinal_mappings.items():
    X_train_encoded[col] = X_train_encoded[col].map(mapping)
    X_test_encoded[col] = X_test_encoded[col].map(mapping)

# Apply Year_study mapping
X_train_encoded["Year_study"] = (
    X_train_encoded["Year_study"].map(year_study_mapping)
)

X_test_encoded["Year_study"] = (
    X_test_encoded["Year_study"].map(year_study_mapping)
)

print("=" * 70)
print("ORDINAL TRANSFORMATION COMPLETE")
print("=" * 70)

print("✓ Ordinal variables transformed")
print("✓ Year_study transformed")

ORDINAL TRANSFORMATION COMPLETE
✓ Ordinal variables transformed
✓ Year_study transformed


In [60]:
# ============================================================
#CELL 22 VALIDATE ORDINAL TRANSFORMATIONS
# ============================================================

print("=" * 70)
print("ORDINAL TRANSFORMATION VALIDATION")
print("=" * 70)

for col in ordinal_cols:
    train_values = sorted(X_train_encoded[col].dropna().unique())
    test_values = sorted(X_test_encoded[col].dropna().unique())

    print(f"\n{col}")
    print(f"  Train: {train_values}")
    print(f"  Test:  {test_values}")

print("\nYear_study")
print(f"  Train: {sorted(X_train_encoded['Year_study'].dropna().unique())}")
print(f"  Test:  {sorted(X_test_encoded['Year_study'].dropna().unique())}")

ORDINAL TRANSFORMATION VALIDATION

ASSIGN_LATE
  Train: [np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
  Test:  [np.int64(1), np.int64(2), np.int64(3), np.int64(4)]

Age_group
  Train: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
  Test:  [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]

SES
  Train: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
  Test:  [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]

Financial_diff
  Train: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
  Test:  [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]

Employment_hrs
  Train: [np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
  Test:  [np.int64(1), np.int64(2), np.int64(3), np.int64(4)]

Study_hrs_day
  Train: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
  Test:  [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]

Sleep_hrs
  Train: [np.int64(1), np.int64

In [61]:
# ============================================================
# CELL 23 -ONE-HOT ENCODING — NOMINAL VARIABLES
# ============================================================

from sklearn.preprocessing import OneHotEncoder

onehot_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False,
    dtype=np.int64
)

# Fit ONLY on training data
onehot_encoder.fit(X_train_encoded[nominal_cols])

# Transform train and test
train_nominal_encoded = onehot_encoder.transform(
    X_train_encoded[nominal_cols]
)

test_nominal_encoded = onehot_encoder.transform(
    X_test_encoded[nominal_cols]
)

# Obtain generated feature names
nominal_feature_names = onehot_encoder.get_feature_names_out(
    nominal_cols
)

print("=" * 70)
print("ONE-HOT ENCODING")
print("=" * 70)

print(f"Original nominal variables: {len(nominal_cols)}")
print(f"Generated dummy variables:  {len(nominal_feature_names)}")

print("\nGenerated features:")
for feature in nominal_feature_names:
    print(f"  - {feature}")

ONE-HOT ENCODING
Original nominal variables: 2
Generated dummy variables:  4

Generated features:
  - Gender_Female
  - Gender_Male
  - Programme_ENT
  - Programme_Nursing


In [62]:
# ============================================================
# CELL 24 - 24CONSTRUCT FINAL ENCODED FEATURE MATRICES
# ============================================================

# Features that remain numeric after preprocessing
numeric_cols = (
    continuous_cols
    + ordinal_cols
    + likert_cols
    + [year_study_col]
)

# Extract numeric components
X_train_numeric = X_train_encoded[numeric_cols].copy()
X_test_numeric = X_test_encoded[numeric_cols].copy()

# Convert one-hot arrays to DataFrames
X_train_nominal_df = pd.DataFrame(
    train_nominal_encoded,
    columns=nominal_feature_names,
    index=X_train_encoded.index
)

X_test_nominal_df = pd.DataFrame(
    test_nominal_encoded,
    columns=nominal_feature_names,
    index=X_test_encoded.index
)

# Combine numeric and one-hot features
X_train_final = pd.concat(
    [X_train_numeric, X_train_nominal_df],
    axis=1
)

X_test_final = pd.concat(
    [X_test_numeric, X_test_nominal_df],
    axis=1
)

print("=" * 70)
print("FINAL ENCODED FEATURE MATRICES")
print("=" * 70)

print(f"X_train_final: {X_train_final.shape}")
print(f"X_test_final:  {X_test_final.shape}")

FINAL ENCODED FEATURE MATRICES
X_train_final: (468, 61)
X_test_final:  (117, 61)


In [63]:
# ============================================================
# CELL 25 -FINAL FEATURE COUNT VALIDATION
# ============================================================

expected_final_features = 61

print("=" * 70)
print("FINAL FEATURE COUNT VALIDATION")
print("=" * 70)

print(f"Expected features: {expected_final_features}")
print(f"Training features: {X_train_final.shape[1]}")
print(f"Testing features:  {X_test_final.shape[1]}")

if (
    X_train_final.shape[1] == expected_final_features
    and X_test_final.shape[1] == expected_final_features
):
    print("\n✓ FINAL FEATURE COUNT VALIDATED")
else:
    raise ValueError(
        "Final feature count does not match expected 61 features."
    )

FINAL FEATURE COUNT VALIDATION
Expected features: 61
Training features: 61
Testing features:  61

✓ FINAL FEATURE COUNT VALIDATED


In [64]:
# ============================================================
# CELL 26 --VERIFY TRAIN / TEST FEATURE ALIGNMENT
# ============================================================

print("=" * 70)
print("TRAIN / TEST FEATURE ALIGNMENT")
print("=" * 70)

train_features = list(X_train_final.columns)
test_features = list(X_test_final.columns)

if train_features == test_features:
    print("✓ Training and testing feature columns are identical")
else:
    print("✗ Feature mismatch detected")

    train_only = set(train_features) - set(test_features)
    test_only = set(test_features) - set(train_features)

    if train_only:
        print("\nTraining-only features:")
        for col in sorted(train_only):
            print(f"  - {col}")

    if test_only:
        print("\nTesting-only features:")
        for col in sorted(test_only):
            print(f"  - {col}")

    raise ValueError("Training and testing feature columns do not align.")

TRAIN / TEST FEATURE ALIGNMENT
✓ Training and testing feature columns are identical


In [65]:
# ============================================================
# CELL 27-- FINAL PREPROCESSING VALIDATION
# ============================================================

print("=" * 70)
print("FINAL PREPROCESSING VALIDATION")
print("=" * 70)

print("\nTraining:")
print(f"  Shape: {X_train_final.shape}")
print(f"  Missing cells: {X_train_final.isnull().sum().sum()}")
print(f"  Non-numeric columns: {X_train_final.select_dtypes(exclude=np.number).shape[1]}")

print("\nTesting:")
print(f"  Shape: {X_test_final.shape}")
print(f"  Missing cells: {X_test_final.isnull().sum().sum()}")
print(f"  Non-numeric columns: {X_test_final.select_dtypes(exclude=np.number).shape[1]}")

if X_train_final.isnull().sum().sum() != 0:
    raise ValueError("Missing values detected in final training predictors.")

if X_test_final.isnull().sum().sum() != 0:
    raise ValueError("Missing values detected in final testing predictors.")

if X_train_final.select_dtypes(exclude=np.number).shape[1] != 0:
    raise ValueError("Non-numeric columns remain in training predictors.")

if X_test_final.select_dtypes(exclude=np.number).shape[1] != 0:
    raise ValueError("Non-numeric columns remain in testing predictors.")

print("\n✓ FINAL PREPROCESSING VALIDATION PASSED")

FINAL PREPROCESSING VALIDATION

Training:
  Shape: (468, 61)
  Missing cells: 0
  Non-numeric columns: 0

Testing:
  Shape: (117, 61)
  Missing cells: 0
  Non-numeric columns: 0

✓ FINAL PREPROCESSING VALIDATION PASSED


In [66]:
# ============================================================
#CELL -28 FINAL BINARY MODELING DATASETS
# ============================================================

train_binary_processed = X_train_final.copy()
test_binary_processed = X_test_final.copy()

train_binary_processed["RISK_BINARY"] = y_train_binary.values
test_binary_processed["RISK_BINARY"] = y_test_binary.values

print("=" * 70)
print("FINAL BINARY DATASETS")
print("=" * 70)

print(f"Training: {train_binary_processed.shape}")
print(f"Testing:  {test_binary_processed.shape}")

FINAL BINARY DATASETS
Training: (468, 62)
Testing:  (117, 62)


In [67]:
# ============================================================
# CELL 29 -FINAL MULTICLASS MODELING DATASETS
# ============================================================

train_multiclass_processed = X_train_final.copy()
test_multiclass_processed = X_test_final.copy()

train_multiclass_processed["RISK_LABEL"] = y_train_multiclass.values
test_multiclass_processed["RISK_LABEL"] = y_test_multiclass.values

print("=" * 70)
print("FINAL MULTICLASS DATASETS")
print("=" * 70)

print(f"Training: {train_multiclass_processed.shape}")
print(f"Testing:  {test_multiclass_processed.shape}")

FINAL MULTICLASS DATASETS
Training: (468, 62)
Testing:  (117, 62)


In [68]:
# ============================================================
# CELL 30--TARGET DISTRIBUTION VALIDATION
# ============================================================

print("=" * 70)
print("BINARY TARGET DISTRIBUTION")
print("=" * 70)

print("\nTraining:")
print(y_train_binary.value_counts().sort_index())

print("\nTesting:")
print(y_test_binary.value_counts().sort_index())


print("\n" + "=" * 70)
print("MULTICLASS TARGET DISTRIBUTION")
print("=" * 70)

print("\nTraining:")
print(y_train_multiclass.value_counts().sort_index())

print("\nTesting:")
print(y_test_multiclass.value_counts().sort_index())

BINARY TARGET DISTRIBUTION

Training:
RISK_BINARY
0    244
1    224
Name: count, dtype: int64

Testing:
RISK_BINARY
0    61
1    56
Name: count, dtype: int64

MULTICLASS TARGET DISTRIBUTION

Training:
RISK_LABEL
High Risk        135
Low Risk         244
Moderate Risk     89
Name: count, dtype: int64

Testing:
RISK_LABEL
High Risk        25
Low Risk         61
Moderate Risk    31
Name: count, dtype: int64


In [69]:
# ============================================================
# CELL 31 --FINAL TARGET LEAKAGE CHECK
# ============================================================

forbidden_predictors = {
    "RISK_BINARY",
    "RISK_LABEL",
    "SID",
    "CGPA",
    "FAIL_COUNT",
    "WARN_STATUS"
}

remaining_forbidden = (
    set(X_train_final.columns)
    & forbidden_predictors
)

print("=" * 70)
print("FINAL TARGET / LEAKAGE CHECK")
print("=" * 70)

print(f"Forbidden variables found in predictors: {remaining_forbidden}")

if remaining_forbidden:
    raise ValueError(
        f"Potential target/leakage variables remain: {remaining_forbidden}"
    )

print("\n✓ NO TARGET, IDENTIFIER, OR KNOWN POST-OUTCOME VARIABLES")
print("  ARE PRESENT IN THE FINAL PREDICTOR MATRIX")

FINAL TARGET / LEAKAGE CHECK
Forbidden variables found in predictors: set()

✓ NO TARGET, IDENTIFIER, OR KNOWN POST-OUTCOME VARIABLES
  ARE PRESENT IN THE FINAL PREDICTOR MATRIX


In [70]:
# ============================================================
# CELL 32 -FINAL NOTEBOOK 2 DIMENSION VALIDATION
# ============================================================

expected_final_shapes = {
    "train_binary_processed": (468, 62),
    "test_binary_processed": (117, 62),
    "train_multiclass_processed": (468, 62),
    "test_multiclass_processed": (117, 62)
}

actual_final_shapes = {
    "train_binary_processed": train_binary_processed.shape,
    "test_binary_processed": test_binary_processed.shape,
    "train_multiclass_processed": train_multiclass_processed.shape,
    "test_multiclass_processed": test_multiclass_processed.shape
}

print("=" * 70)
print("FINAL NOTEBOOK 2 DIMENSION VALIDATION")
print("=" * 70)

for name, expected in expected_final_shapes.items():

    actual = actual_final_shapes[name]

    if actual == expected:
        print(f"✓ {name:<30} {actual}")
    else:
        print(
            f"✗ {name:<30} "
            f"Expected {expected}, got {actual}"
        )
        raise ValueError(
            f"{name} has unexpected dimensions."
        )

print("\n✓ ALL FINAL DATASET DIMENSIONS VALIDATED")

FINAL NOTEBOOK 2 DIMENSION VALIDATION
✓ train_binary_processed         (468, 62)
✓ test_binary_processed          (117, 62)
✓ train_multiclass_processed     (468, 62)
✓ test_multiclass_processed      (117, 62)

✓ ALL FINAL DATASET DIMENSIONS VALIDATED


In [71]:
# ============================================================
#CELL 33- -FINAL INTEGRITY CHECK
# ============================================================

print("=" * 70)
print("FINAL NOTEBOOK 2 INTEGRITY CHECK")
print("=" * 70)


# ------------------------------------------------------------
# 1. Check shapes
# ------------------------------------------------------------

assert train_binary.shape == (468, 62)
assert test_binary.shape == (117, 62)
assert train_multiclass.shape == (468, 62)
assert test_multiclass.shape == (117, 62)

print("✓ Dataset dimensions correct")


# ------------------------------------------------------------
# 2. Check missing values
# ------------------------------------------------------------

assert train_binary.isnull().sum().sum() == 0
assert test_binary.isnull().sum().sum() == 0
assert train_multiclass.isnull().sum().sum() == 0
assert test_multiclass.isnull().sum().sum() == 0

print("✓ No missing values")


# ------------------------------------------------------------
# 3. Check predictors are numeric
# ------------------------------------------------------------

assert X_train_final.select_dtypes(exclude=np.number).shape[1] == 0
assert X_test_final.select_dtypes(exclude=np.number).shape[1] == 0

print("✓ All predictors are numeric")


# ------------------------------------------------------------
# 4. Check forbidden variables
# ------------------------------------------------------------

forbidden_predictors = {
    "SID",
    "RISK_BINARY",
    "RISK_LABEL",
    "CGPA",
    "FAIL_COUNT",
    "WARN_STATUS"
}

assert not (
    set(X_train_final.columns) & forbidden_predictors
)

print("✓ No identifier, target, or known leakage variables")


# ------------------------------------------------------------
# 5. Check train/test feature alignment
# ------------------------------------------------------------

assert list(X_train_final.columns) == list(X_test_final.columns)

print("✓ Training/testing feature alignment")


# ------------------------------------------------------------
# 6. Check target lengths
# ------------------------------------------------------------

assert len(y_train_binary) == len(X_train_final)
assert len(y_test_binary) == len(X_test_final)

assert len(y_train_multiclass) == len(X_train_final)
assert len(y_test_multiclass) == len(X_test_final)

print("✓ Target/predictor row alignment")


print("=" * 70)
print("✓ NOTEBOOK 2 INTEGRITY CHECK PASSED")
print("=" * 70)

FINAL NOTEBOOK 2 INTEGRITY CHECK
✓ Dataset dimensions correct
✓ No missing values
✓ All predictors are numeric
✓ No identifier, target, or known leakage variables
✓ Training/testing feature alignment
✓ Target/predictor row alignment
✓ NOTEBOOK 2 INTEGRITY CHECK PASSED


In [72]:
# ============================================================
# CELL 34 -SAVE FINAL NOTEBOOK 2 DATASETS
# ============================================================

train_binary_processed.to_csv(
    PROCESSED_DIR / "train_binary_processed.csv",
    index=False
)

test_binary_processed.to_csv(
    PROCESSED_DIR / "test_binary_processed.csv",
    index=False
)

train_multiclass_processed.to_csv(
    PROCESSED_DIR / "train_multiclass_processed.csv",
    index=False
)

test_multiclass_processed.to_csv(
    PROCESSED_DIR / "test_multiclass_processed.csv",
    index=False
)

print("=" * 70)
print("NOTEBOOK 2 DATASETS SAVED")
print("=" * 70)

print(f"✓ {PROCESSED_DIR / 'train_binary_processed.csv'}")
print(f"✓ {PROCESSED_DIR / 'test_binary_processed.csv'}")
print(f"✓ {PROCESSED_DIR / 'train_multiclass_processed.csv'}")
print(f"✓ {PROCESSED_DIR / 'test_multiclass_processed.csv'}")

NOTEBOOK 2 DATASETS SAVED
✓ ..\data\processed\train_binary_processed.csv
✓ ..\data\processed\test_binary_processed.csv
✓ ..\data\processed\train_multiclass_processed.csv
✓ ..\data\processed\test_multiclass_processed.csv


In [73]:
# ============================================================
# CELL 35 --RELOAD VERIFICATION
# ============================================================

check_train_binary = pd.read_csv(
    PROCESSED_DIR / "train_binary_processed.csv"
)

check_test_binary = pd.read_csv(
    PROCESSED_DIR / "test_binary_processed.csv"
)

check_train_multiclass = pd.read_csv(
    PROCESSED_DIR / "train_multiclass_processed.csv"
)

check_test_multiclass = pd.read_csv(
    PROCESSED_DIR / "test_multiclass_processed.csv"
)

print("=" * 70)
print("RELOAD VERIFICATION")
print("=" * 70)

print(f"Binary training:     {check_train_binary.shape}")
print(f"Binary testing:      {check_test_binary.shape}")
print(f"Multiclass training: {check_train_multiclass.shape}")
print(f"Multiclass testing:  {check_test_multiclass.shape}")

assert check_train_binary.shape == (468, 62)
assert check_test_binary.shape == (117, 62)
assert check_train_multiclass.shape == (468, 62)
assert check_test_multiclass.shape == (117, 62)

print("\n✓ SAVED FILES RELOADED SUCCESSFULLY")
print("✓ NOTEBOOK 2 COMPLETE")

RELOAD VERIFICATION
Binary training:     (468, 62)
Binary testing:      (117, 62)
Multiclass training: (468, 62)
Multiclass testing:  (117, 62)

✓ SAVED FILES RELOADED SUCCESSFULLY
✓ NOTEBOOK 2 COMPLETE
